In [ ]:
%pip install -q -U "transformers>=5.12" accelerate bitsandbytes huggingface_hub "pillow<12"

## Local Inference on GPU 
Model page: https://huggingface.co/orcarouter/Qwen3.8-27B-Uncensored

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/orcarouter/Qwen3.8-27B-Uncensored)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
from huggingface_hub import login
login()

In [ ]:
# Load one 4-bit model across the available T4 GPUs.
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "orcarouter/Qwen3.8-27B-Uncensored"
assert torch.cuda.is_available(), "Select GPU T4 x2 in Session options."
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    max_memory={i: "14GiB" for i in range(torch.cuda.device_count())},
    dtype=torch.float16,
    attn_implementation="sdpa",
)
model.eval()
print("Model loaded:", model.hf_device_map)

In [ ]:
# Run this cell again to ask another question; the model stays loaded.
import gc
import requests
from PIL import Image
from io import BytesIO


gc.collect()
torch.cuda.empty_cache()
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
response = requests.get(image_url, timeout=60)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")
image.thumbnail((384, 384))  # Keep vision attention within T4 memory limits.
messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "What animal is on the candy? Answer in one short sentence."},
]}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
for key, value in inputs.items():
    if torch.is_tensor(value) and value.is_floating_point():
        inputs[key] = value.to(torch.float16)
print("Image size:", image.size, "Input tokens:", inputs["input_ids"].shape[-1])
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

In [ ]:
%pip install -q fastapi uvicorn pyngrok
import fastapi, uvicorn, pyngrok
print("API dependencies ready")

In [ ]:
# OpenAI-compatible text API: reuses the loaded 4-bit model.
import json, time, uuid, threading, queue, gc
from typing import Any, Literal
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from transformers import TextIteratorStreamer, StoppingCriteria, StoppingCriteriaList
import uvicorn

API_MODEL = MODEL_ID
api_tokenizer = processor.tokenizer
api_lock = threading.Lock()
app = FastAPI(title="Kaggle Qwen 27B API")

class ChatRequest(BaseModel):
    model: str = API_MODEL
    messages: list[dict[str, Any]] = Field(min_length=1, max_length=100)
    max_tokens: int = Field(default=256, ge=1, le=512)
    temperature: float = Field(default=0.6, ge=0, le=2)
    top_p: float = Field(default=0.95, gt=0, le=1)
    stream: bool = False
    stop: str | list[str] | None = None
    tools: list[dict[str, Any]] | None = None
    tool_choice: Literal["auto", "none"] = "auto"

@app.get("/")
@app.get("/health")
def health():
    return {"status": "ok", "model": API_MODEL, "busy": api_lock.locked()}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": API_MODEL, "object": "model", "created": int(time.time()), "owned_by": "local"}]}

class CancelGeneration(StoppingCriteria):
    def __init__(self, event):
        self.event = event
    def __call__(self, input_ids, scores, **kwargs):
        return self.event.is_set()

def parse_calls(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[index:])
            calls = value.get("tool_calls", [])
            if not calls:
                continue
            result = []
            for call in calls:
                function = call.get("function", call)
                arguments = function.get("arguments", {})
                result.append({"id": "call_" + uuid.uuid4().hex[:12], "type": "function", "function": {
                    "name": function["name"],
                    "arguments": arguments if isinstance(arguments, str) else json.dumps(arguments),
                }})
            return result
        except (ValueError, KeyError, AttributeError, TypeError):
            continue
    return []

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    if req.model != API_MODEL:
        raise HTTPException(404, "Unknown model. See /v1/models.")
    stops = [req.stop] if isinstance(req.stop, str) else (req.stop or [])
    if len(stops) > 4 or any(not s or len(s) > 200 for s in stops):
        raise HTTPException(422, "Use up to four nonempty stop strings, each at most 200 characters.")
    messages = []
    for message in req.messages:
        if message.get("role") not in {"system", "user", "assistant", "tool"}:
            raise HTTPException(422, "Unsupported message role.")
        content = message.get("content") or ""
        if not isinstance(content, str):
            raise HTTPException(422, "This endpoint accepts text messages only.")
        if message.get("tool_calls"):
            content += json.dumps({"tool_calls": message["tool_calls"]})
        messages.append({"role": message["role"], "content": content})
    use_tools = bool(req.tools) and req.tool_choice != "none"
    if use_tools:
        instruction = ('Available tools: ' + json.dumps(req.tools) +
            '\nWhen a tool is needed, respond ONLY with JSON in this format: '
            '{"tool_calls":[{"name":"function_name","arguments":{"key":"value"}}]}. '
            'Otherwise answer normally. Tool results are provided in tool messages.')
        messages.insert(0, {"role": "system", "content": instruction})
    if sum(len(m["content"]) for m in messages) > 20000:
        raise HTTPException(413, "Request is too long for this T4 demo.")
    encoded = api_tokenizer.apply_chat_template(messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt", enable_thinking=False)
    prompt_tokens = encoded["input_ids"].shape[-1]
    if prompt_tokens > 2048:
        raise HTTPException(413, "Maximum input length is 2048 tokens on this T4 demo.")
    if not api_lock.acquire(blocking=False):
        raise HTTPException(429, "Model is busy; retry after the current request finishes.")
    cancel = threading.Event()
    state = {}
    try:
        inputs = encoded.to(model.device)
        streamer = TextIteratorStreamer(api_tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=1)
        options = dict(max_new_tokens=req.max_tokens, do_sample=req.temperature > 0,
            streamer=streamer, stopping_criteria=StoppingCriteriaList([CancelGeneration(cancel)]))
        if req.temperature > 0:
            options.update(temperature=req.temperature, top_p=req.top_p)
        if stops:
            options.update(stop_strings=stops, tokenizer=api_tokenizer)
        def worker():
            try:
                with torch.inference_mode():
                    generated = model.generate(**inputs, **options)
                state["tokens"] = generated.shape[-1] - prompt_tokens
            except Exception as exc:
                state["error"] = type(exc).__name__
                streamer.end()
            finally:
                api_lock.release()
        thread = threading.Thread(target=worker, daemon=True)
        thread.start()
    except Exception:
        api_lock.release()
        raise
    request_id, created = "chatcmpl-" + uuid.uuid4().hex[:16], int(time.time())
    base = {"id": request_id, "created": created, "model": API_MODEL}
    def pieces():
        pending = ""
        hold = max(map(len, stops), default=0)
        try:
            while True:
                try:
                    piece = next(streamer)
                except queue.Empty:
                    if not thread.is_alive():
                        break
                    continue
                except StopIteration:
                    break
                pending += piece
                matches = [pending.index(s) for s in stops if s in pending]
                if matches:
                    yield pending[:min(matches)]
                    cancel.set()
                    pending = ""
                    break
                safe = len(pending) - hold
                if safe > 0:
                    yield pending[:safe]
                    pending = pending[safe:]
            if pending:
                yield pending
            thread.join()
            if "error" in state:
                raise RuntimeError(state["error"])
        finally:
            cancel.set()
    def finish():
        return "length" if state.get("tokens", 0) >= req.max_tokens else "stop"
    if not req.stream:
        try:
            text = "".join(pieces())
        except RuntimeError as exc:
            raise HTTPException(500, "Generation failed: " + str(exc))
        calls = parse_calls(text) if use_tools else []
        message = {"role": "assistant", "content": None if calls else text}
        if calls:
            message["tool_calls"] = calls
        completion = state.get("tokens", 0)
        return {**base, "object": "chat.completion", "choices": [{"index": 0, "message": message,
            "finish_reason": "tool_calls" if calls else finish()}],
            "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion, "total_tokens": prompt_tokens + completion}}
    def event(delta, reason=None):
        return "data: " + json.dumps({**base, "object": "chat.completion.chunk", "choices": [
            {"index": 0, "delta": delta, "finish_reason": reason}]}) + "\n\n"
    def events():
        try:
            yield event({"role": "assistant"})
            if use_tools:
                text = "".join(pieces())
                calls = parse_calls(text)
                if calls:
                    yield event({"tool_calls": [{"index": i, **call} for i, call in enumerate(calls)]})
                else:
                    yield event({"content": text})
                yield event({}, "tool_calls" if calls else finish())
            else:
                for piece in pieces():
                    if piece:
                        yield event({"content": piece})
                yield event({}, finish())
            yield "data: [DONE]\n\n"
        except Exception as exc:
            yield "data: " + json.dumps({"error": {"message": "Generation failed: " + type(exc).__name__}}) + "\n\n"
        finally:
            cancel.set()
    return StreamingResponse(events(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

# Bind only the API to loopback; ngrok will forward to this port.
if "api_server" in globals():
    api_server.should_exit = True
    api_thread.join(timeout=10)
api_server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
api_thread = threading.Thread(target=api_server.run, daemon=True)
api_thread.start()
for _ in range(100):
    if api_server.started:
        break
    time.sleep(0.1)
assert api_server.started, "API server failed to start"
print("API ready at http://127.0.0.1:8000; model:", API_MODEL)


In [ ]:
# Verify chat, real SSE streaming, stop handling, and tool-call formatting.
import requests, json
base_url = "http://127.0.0.1:8000"
assert requests.get(base_url + "/health", timeout=10).json()["status"] == "ok"
assert requests.get(base_url + "/v1/models", timeout=10).json()["data"][0]["id"] == API_MODEL
payload = {"model": API_MODEL, "messages": [{"role": "user", "content": "What is 2 + 2? Reply with only the number."}], "max_tokens": 32, "temperature": 0}
r = requests.post(base_url + "/v1/chat/completions", json=payload, timeout=180)
r.raise_for_status()
print("CHAT:", r.json()["choices"][0]["message"])
with requests.post(base_url + "/v1/chat/completions", json={**payload, "stream": True}, stream=True, timeout=180) as r:
    r.raise_for_status()
    chunks, done = [], False
    for line in r.iter_lines():
        if line == b"data: [DONE]":
            done = True
        elif line.startswith(b"data: "):
            chunk = json.loads(line[6:])
            assert "error" not in chunk, chunk
            chunks.append(chunk)
    assert done and chunks[-1]["choices"][0]["finish_reason"] in {"stop", "length"}
    print("STREAM:", "".join(c["choices"][0]["delta"].get("content", "") for c in chunks), "DONE:", done)
r = requests.post(base_url + "/v1/chat/completions", json={**payload, "stop": "4"}, timeout=180)
r.raise_for_status()
assert "4" not in (r.json()["choices"][0]["message"]["content"] or "")
print("STOP: passed")
tool_payload = {**payload, "max_tokens": 128, "messages": [{"role": "user", "content": "Use get_weather to check the weather in Paris. Return the tool call only."}], "tools": [{"type": "function", "function": {"name": "get_weather", "description": "Get current weather in a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}]}
r = requests.post(base_url + "/v1/chat/completions", json=tool_payload, timeout=180)
r.raise_for_status()
print("TOOLS:", r.json()["choices"][0])
assert r.json()["choices"][0]["finish_reason"] == "tool_calls"
print("LOCAL API CHECKS PASSED")

In [ ]:
# Authenticate ngrok without putting the token in notebook source.
from pyngrok import ngrok, conf
from getpass import getpass
conf.get_default().auth_token = getpass("ngrok authtoken: ")
print("ngrok token configured")

In [ ]:
# Publish only the model API through ngrok (no API key, matching the Colab demo).
assert api_server.started, "Run the API server cell first"
if "tunnel" in globals():
    ngrok.disconnect(tunnel.public_url)
tunnel = ngrok.connect(addr="http://127.0.0.1:8000", proto="http", bind_tls=True)
public_url = tunnel.public_url
print("PUBLIC BASE URL:", public_url + "/v1")
print("MODEL:", API_MODEL)
print("DOCS:", public_url + "/docs")
print("API key for OpenAI clients: not-needed")
print("Limits: text only; 2048 input tokens; 512 output tokens; one request at a time.")
# Verify a full request through the public tunnel.
check = requests.post(public_url + "/v1/chat/completions",
    headers={"ngrok-skip-browser-warning": "true"},
    json={"model": API_MODEL, "messages": [{"role": "user", "content": "What is 2+2? Reply with only the number."}], "temperature": 0, "max_tokens": 16},
    timeout=180)
check.raise_for_status()
print("PUBLIC TEST:", check.json()["choices"][0]["message"]["content"])
# To stop public access, run: ngrok.disconnect(tunnel.public_url)
# To stop the local API too, run: api_server.should_exit = True
